# Module 04: Web Search & Browsing

This notebook is the **lab**. The lesson is [`outline.md`](outline.md).

Closed weights cannot answer “what shipped this week.” Retrieval can. The web is one retrieval store (alongside your docs, your DB, MCP). Pattern:

**search → select → visit → extract → cite**

We use `WebSearchTool` (current smolagents default) and `VisitWebpageTool`. Isolation-test both before any agent.

This module **will flake**. Rate limits and bot walls are part of the lesson, not a failed install.

Errors: [`instructions.md`](instructions.md).


## Setup

**Local (canonical):** this cell imports `course_setup.py` from the repo root. Your `.env` must live next to `pyproject.toml`.

**Colab:** run the two *Colab only* cells first (install + secrets), then this one. The fallback path uses `HF_TOKEN` from the environment.


In [ ]:
# Colab only — skip this cell locally.
# !pip install -q "smolagents[toolkit,litellm]" python-dotenv pandas requests markdownify huggingface-hub mlflow


In [ ]:
# Colab only — skip this cell locally.
# In Colab: Secrets (key icon) → add HF_TOKEN with "Make calls to Inference Providers".
# import os
# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


In [ ]:
import os
import sys
from pathlib import Path

def _repo_root() -> Path | None:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "course_setup.py").exists():
            return candidate
    return None

ROOT = _repo_root()
if ROOT is not None:
    sys.path.insert(0, str(ROOT))
    from course_setup import make_model, print_setup, smoke_test

    model = make_model()
    print_setup(model)
    smoke_test(model)
else:
    from dotenv import load_dotenv
    from smolagents import InferenceClientModel

    load_dotenv()
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError(
            "course_setup.py was not found and HF_TOKEN is unset. "
            "Local: start Jupyter from the cloned repo (`uv run jupyter lab`). "
            "Colab: run the secrets cell, then re-run this cell."
        )
    model = InferenceClientModel(
        model_id=os.environ.get("COURSE_MODEL_ID", "Qwen/Qwen3-Next-80B-A3B-Thinking"),
        token=token,
    )
    print("Model initialized:", model.model_id)


## Isolation test: search

`engine="duckduckgo"` is the default. Also `"bing"` and `"exa"` (Exa needs `EXA_API_KEY`).


In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, VisitWebpageTool, WebSearchTool

search = WebSearchTool(max_results=5, engine="duckduckgo")
print("schema:", search.name, search.inputs, "engine:", search.engine)

try:
    raw = search("smolagents huggingface documentation")
except Exception as exc:
    print("duckduckgo engine failed:", type(exc).__name__, exc)
    print("Falling back to engine='bing' — this is expected on some networks.")
    search = WebSearchTool(max_results=5, engine="bing")
    raw = search("smolagents huggingface documentation")
print(raw[:1500])


## Isolation test: visit

If this returns a CAPTCHA or nearly nothing, the host blocked you. Try another URL. Truncation is normal on long docs.


In [ ]:
visit = VisitWebpageTool(max_output_length=8000)
page = visit("https://huggingface.co/docs/smolagents")
print(page[:1200])
print("\n... length", len(page))


## Agent: search only

Useful for “give me links.” Snippets are not the page — notice how little detail you get.


In [ ]:
search_agent = CodeAgent(
    tools=[search],
    model=model,
    max_steps=6,
)
r = search_agent.run(
    "Find the official smolagents documentation URL on Hugging Face. "
    "Return the URL you found in the search results, not one you invent."
)
print(r)
print("steps", [type(s).__name__ for s in search_agent.memory.steps])


## Agent: search + visit + cite

The task **requires** a visit and a URL. After it runs, skim `memory.steps` and check the URL actually appeared in an observation.


In [ ]:
research_agent = CodeAgent(
    tools=[search, visit],
    model=model,
    max_steps=8,
)
task = (
    "What is Hugging Face smolagents? Visit the official docs or blog, "
    "extract a one-sentence description, and cite the exact URL you visited. "
    "Only cite a URL that appeared in a tool observation."
)
r = research_agent.run(task)
print(r)
print("steps", [type(s).__name__ for s in research_agent.memory.steps])


In [ ]:
print("Did a visit happen?")
for i, step in enumerate(research_agent.memory.steps):
    calls = getattr(step, "tool_calls", None)
    if calls:
        print(i, calls)


## Exercises

Exercise 2 is **stretch**. If search rate-limits you, stop after one source and write what you would change.


In [ ]:
# TODO Exercise 1: latest stable Python
# Agent with WebSearchTool + VisitWebpageTool, max_steps=8.
# Task: latest stable Python release. Prefer python.org. Visit the page.
# Extract the exact version (e.g. 3.13.x). Cite the URL.
#
# You succeeded if:
#   - the result contains a version number that looks like 3.xx
#   - you can point to a visit_webpage (or equivalent) in memory.steps
#     OR you honestly note that the site blocked you and you used snippets
#   - the cited URL also appears in an observation, not only in the final sentence

# Your code here:


In [ ]:
# TODO Exercise 2 (stretch): mini research table
# Same tools, max_steps=10.
# Pick a topic you can verify (default: "dbt data build tool best practices 2025 2026").
# Return a markdown table: Title | One takeaway | URL  (up to 3 rows).
# If you get rate-limited, return 1 row and a sentence on what you would change.
#
# You succeeded if:
#   - at least one row has a URL that appeared in the trace
#   - you did not invent a comparison of 3 articles from snippets alone
#     without saying so

# Your code here:


## Hints

<details>
<summary>Exercise 1 prompt shape</summary>

“What is the latest stable Python version? Visit https://www.python.org/downloads/ (or the URL search returns) and extract the version from the page. Cite that URL.”

</details>

<details>
<summary>Rate limited</summary>

`time.sleep(5)` between runs, or `WebSearchTool(engine="bing")`. Do not tight-loop the same query.

</details>


## What you built

- A retrieval loop that does not depend on smolagents: search, visit, extract, cite
- Explicit `WebSearchTool` + `VisitWebpageTool` instead of a mystery `add_base_tools`
- A habit of checking that cited URLs existed in observations

**Key insight:** the web is just another tool-shaped store. Snippets are advertisements for pages. Read the page.

---

## Next — Module 05: Multi-agent orchestration

One agent holding search *and* analysis *and* writing will thrash. A manager with named specialists — **descriptions as the routing table** — is the next pattern.
